# Cortex AI ハンズオン — コスメECサイト

このハンズオンでは、Snowflake の Cortex AI 機能を使ってコスメECサイトについて自然言語で質問できる AI エージェント作成までを一気通貫で構築します。

完成すると、例えばこんな質問に自然言語で答えてくれます：
- 「品質・効果の観点でポジティブ評価が多いブランドはどこですか？」
- 「開封済みの商品は返品できる？」
- 「在庫が少ない商品のうち、レビュー評価が高いものはどれ？」

## ハンズオンの流れ

| Section | やること | 学べる Snowflake 機能 |
|:-------:|----------|----------------------|
| 1 | AI でデータを拡張する | AI Functions（AI_CLASSIFY / AI_COMPLETE） |
| 2 | FAQ・商品・レビューのセマンティック検索サービスを構築する | Cortex Search Service |
| 3 | 売上分析・レビュー分析用のセマンティックビューを構築する | Semantic View（Cortex Analyst） |
| 4 | AI エージェントを組み立てる | Cortex Agent |
| 5 | 動かしてみよう！ | Snowflake Intelligence |
| 6 | エージェントを評価する | Cortex Agent Evaluations |

## Step 0: 事前準備

使用するデータベース・スキーマ・ウェアハウスを設定します。

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COSME_EC_WH;
USE DATABASE COSME_EC_HANDSON;
USE SCHEMA ANALYTICS;

### データの確認

まず最初に、今回操作するデータについて確認していきましょう。

In [ ]:
%%sql -r dataframe_2
-- 商品マスタ（50商品のコスメ商品）
SELECT * FROM COSME_EC_HANDSON.RAW.PRODUCTS LIMIT 10;

In [ ]:
%%sql -r dataframe_3
-- 顧客マスタ（200名の会員情報）
SELECT * FROM COSME_EC_HANDSON.RAW.CUSTOMERS LIMIT 10;

In [ ]:
%%sql -r dataframe_4
-- 注文データ（約1,000件の注文）
SELECT * FROM COSME_EC_HANDSON.RAW.ORDERS LIMIT 10;

In [ ]:
%%sql -r dataframe_5
-- 注文明細（約3,640件の注文明細）
SELECT * FROM COSME_EC_HANDSON.RAW.ORDER_ITEMS LIMIT 10;

In [ ]:
%%sql -r dataframe_6
-- 商品レビュー（500件、日本語）
SELECT * FROM COSME_EC_HANDSON.RAW.REVIEWS LIMIT 10;

In [ ]:
%%sql -r dataframe_7
-- FAQドキュメント（30件のナレッジベース）
SELECT * FROM COSME_EC_HANDSON.RAW.FAQ_DOCS LIMIT 10;

## Section 1: AI-Powered Data Pipeline（AI関数を活用する）

RAW スキーマのデータに Cortex AI 関数で付加情報を追加したテーブルを ANALYTICS スキーマに作成します。

### PRODUCTS_WITH_CATEGORY（AI カテゴリ付き商品マスタ）

商品名と説明文から **AI_CLASSIFY** で 5 つのカテゴリに自動分類します。

| カテゴリ | 対象商品 |
|---------|---------|
| スキンケア | 化粧水、乳液、美容液、クレンジング、洗顔、クリーム等 |
| メイク | ファンデーション、アイシャドウ、リップ、チーク等 |
| ヘアケア | シャンプー、コンディショナー、トリートメント等 |
| ボディケア | ボディソープ、ボディクリーム、ハンドクリーム等 |
| フレグランス | 香水、ボディミスト、ルームフレグランス等 |

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE TABLE COSME_EC_HANDSON.ANALYTICS.PRODUCTS_WITH_CATEGORY
    COMMENT = 'AIカテゴリ付き商品マスタ（RAW.PRODUCTSにAI_CLASSIFYで自動分類したカテゴリを付与）'
AS
SELECT
    *,
    AI_CLASSIFY(
        PRODUCT_NAME || ' - ' || DESCRIPTION,
        [
            {'label': 'スキンケア', 'description': '化粧水、乳液、美容液、クレンジング、洗顔、クリーム、日焼け止め、美容オイル、ミスト化粧水など肌のお手入れ製品'},
            {'label': 'メイク', 'description': 'ファンデーション、アイシャドウ、リップ、チーク、マスカラ、アイブロウ、コンシーラー、フェイスパウダーなど化粧品'},
            {'label': 'ヘアケア', 'description': 'シャンプー、コンディショナー、トリートメント、ヘアオイル、ヘアミスト、頭皮ケアなど髪のお手入れ製品'},
            {'label': 'ボディケア', 'description': 'ボディソープ、ボディクリーム、ボディオイル、ハンドクリーム、入浴剤、リップケア、ネイルケアなど体のお手入れ製品'},
            {'label': 'フレグランス', 'description': '香水、ボディミスト、ルームフレグランス、アロマオイル、練り香水、ヘアフレグランスなど香り製品'}
        ],
        {'task_description': 'コスメ・化粧品の商品名と説明文から最も適切なカテゴリに分類してください'}
    ):labels[0]::VARCHAR AS CATEGORY
FROM COSME_EC_HANDSON.RAW.PRODUCTS;

In [ ]:
%%sql -r dataframe_9
SELECT CATEGORY, COUNT(PRODUCT_ID) AS PROD_CNT 
FROM COSME_EC_HANDSON.ANALYTICS.PRODUCTS_WITH_CATEGORY 
GROUP BY 1 ORDER BY 2 DESC;

### REVIEW_DETAILS（観点別レビュー分析）

各レビューを **AI_COMPLETE** で 5 つの観点に分解し、観点ごとの感情ラベル・スコア・要約を生成します。言及されていない観点は除外されます。

| 観点キー | 日本語名 |
|---------|---------|
| quality | 品質・効果 |
| texture | 使用感・テクスチャー |
| price | 価格・コスパ |
| skin_reaction | 肌トラブル・肌への影響 |
| delivery | 配送・梱包 |

In [ ]:
%%sql -r dataframe_10
CREATE OR REPLACE TABLE COSME_EC_HANDSON.ANALYTICS.REVIEW_DETAILS
    COMMENT = '観点別レビュー分析テーブル（AI_COMPLETEで各レビューを5観点に分解し感情ラベル・スコア・要約を生成）'
AS
WITH called AS (
  SELECT
    r.REVIEW_ID,
    AI_COMPLETE(
      model  => 'claude-sonnet-4-6',
      prompt =>
        'あなたはコスメECサイトのレビュー分析を行うアナリストです。' ||
        '次のレビューについて、各観点ごとに以下の情報を出力してください。' ||
        '\n\n必ず次の5観点を全て含めてください:' ||
        '\n- quality: 品質・効果' ||
        '\n- texture: 使用感・テクスチャー' ||
        '\n- price: 価格・コスパ' ||
        '\n- skin_reaction: 肌トラブル・肌への影響' ||
        '\n- delivery: 配送・梱包' ||
        '\n\n各観点について必ず次のフィールドを出力してください:' ||
        '\n- aspect_key: 上記の英語キーをそのまま (quality / texture / price / skin_reaction / delivery)' ||
        '\n- aspect_name_ja: 日本語の観点名' ||
        '\n    - quality       → 品質・効果' ||
        '\n    - texture       → 使用感・テクスチャー' ||
        '\n    - price         → 価格・コスパ' ||
        '\n    - skin_reaction → 肌トラブル・肌への影響' ||
        '\n    - delivery      → 配送・梱包' ||
        '\n- sentiment_label: positive / negative / neutral / not_mentioned のいずれか' ||
        '\n- sentiment_score: positive=1, neutral=0, negative=-1, not_mentioned=0 とする数値' ||
        '\n- summary: その観点に関する内容の1文要約（日本語）。' ||
        '\n           レビュー内でその観点に一切触れられていない場合は、' ||
        '\n           sentiment_label を not_mentioned とし、summary には必ず空文字列("")を入れること。' ||
        '\n           null や NULL は絶対に使わないこと。' ||
        '\n\nレビュー本文:\n' || r.REVIEW_TEXT,
      response_format => TYPE OBJECT(
        quality       OBJECT(
          aspect_key      STRING,
          aspect_name_ja  STRING,
          sentiment_label STRING,
          sentiment_score NUMBER,
          summary         STRING
        ),
        texture       OBJECT(
          aspect_key      STRING,
          aspect_name_ja  STRING,
          sentiment_label STRING,
          sentiment_score NUMBER,
          summary         STRING
        ),
        price         OBJECT(
          aspect_key      STRING,
          aspect_name_ja  STRING,
          sentiment_label STRING,
          sentiment_score NUMBER,
          summary         STRING
        ),
        skin_reaction OBJECT(
          aspect_key      STRING,
          aspect_name_ja  STRING,
          sentiment_label STRING,
          sentiment_score NUMBER,
          summary         STRING
        ),
        delivery      OBJECT(
          aspect_key      STRING,
          aspect_name_ja  STRING,
          sentiment_label STRING,
          sentiment_score NUMBER,
          summary         STRING
        )
      )
    )::VARIANT AS ASPECT_RESULT
  FROM COSME_EC_HANDSON.RAW.REVIEWS r
),
exploded AS (
  SELECT
    c.REVIEW_ID,
    f.value:aspect_key::STRING      AS ASPECT_KEY,
    f.value:aspect_name_ja::STRING  AS ASPECT_NAME_JA,
    f.value:sentiment_label::STRING AS SENTIMENT_LABEL,
    f.value:sentiment_score::NUMBER AS SENTIMENT_SCORE,
    NULLIF(f.value:summary::STRING, '') AS ASPECT_SUMMARY
  FROM called c,
       LATERAL FLATTEN(input => c.ASPECT_RESULT) f
  WHERE f.value:sentiment_label::STRING <> 'not_mentioned'
)
SELECT * FROM exploded;

In [ ]:
%%sql -r dataframe_11
-- 結果を確認してみましょう。各レビューが観点ごとに分解されています。
SELECT * FROM COSME_EC_HANDSON.ANALYTICS.REVIEW_DETAILS LIMIT 20;

## Section 2: Cortex Search Service（非構造化データの検索サービス）

Cortex Search は、非構造化テキストデータに対してセマンティック検索を可能にするサービスです。
ベクトル埋め込みモデル（`snowflake-arctic-embed-l-v2.0`）を使い、キーワード一致ではなく意味的な類似度で検索します。

### 今回作成する 3 つの Search Service

| サービス名 | データソース | 用途 |
|-----------|------------|------|
| **FAQ_SEARCH** | FAQ_DOCS | 返品ポリシー・配送・ポイント制度など |
| **PRODUCT_SEARCH** | PRODUCTS_WITH_CATEGORY | 商品検索・おすすめ商品の提案 |
| **REVIEW_SEARCH** | REVIEW_DETAILS | レビュー内容の検索・口コミ調査 |

In [ ]:
%%sql -r dataframe_12
CREATE OR REPLACE CORTEX SEARCH SERVICE COSME_EC_HANDSON.ANALYTICS.FAQ_SEARCH
    ON CONTENT
    ATTRIBUTES FAQ_ID, CATEGORY, TITLE, LAST_UPDATED
    WAREHOUSE = COSME_EC_WH
    TARGET_LAG = '1 hour'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        FAQ_ID,
        TITLE,
        CONTENT,
        CATEGORY,
        LAST_UPDATED
    FROM COSME_EC_HANDSON.RAW.FAQ_DOCS
);

In [ ]:
%%sql -r faq_search_result
-- FAQ_SEARCH にクエリ（例：返品ポリシーについて）
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'COSME_EC_HANDSON.ANALYTICS.FAQ_SEARCH',
    '{
       "query": "返品はできますか？",
       "columns": ["FAQ_ID", "TITLE", "CONTENT", "CATEGORY"],
       "limit": 3
    }'
  )
)['results'] AS results;

In [ ]:
%%sql -r dataframe_13
CREATE OR REPLACE CORTEX SEARCH SERVICE COSME_EC_HANDSON.ANALYTICS.PRODUCT_SEARCH
    ON SEARCH_TEXT
    ATTRIBUTES PRODUCT_ID, PRODUCT_NAME, BRAND, CATEGORY
    WAREHOUSE = COSME_EC_WH
    TARGET_LAG = '1 hour'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        PRODUCT_ID,
        PRODUCT_NAME,
        BRAND,
        DESCRIPTION,
        CATEGORY,
        CONCAT(
            COALESCE(PRODUCT_NAME, ''), ' ',
            COALESCE(BRAND, ''), ' ',
            COALESCE(CATEGORY, ''), ' ',
            COALESCE(DESCRIPTION, ''),
            CASE WHEN IS_ORGANIC THEN ' オーガニック 自然派 天然成分' ELSE '' END
        ) AS SEARCH_TEXT
    FROM COSME_EC_HANDSON.ANALYTICS.PRODUCTS_WITH_CATEGORY
);

In [ ]:
%%sql -r product_search_result
-- PRODUCT_SEARCH にクエリ（例：敏感肌向け商品を検索）
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'COSME_EC_HANDSON.ANALYTICS.PRODUCT_SEARCH',
    '{
       "query": "敏感肌向けのスキンケア",
       "columns": ["PRODUCT_ID", "PRODUCT_NAME", "BRAND", "CATEGORY"],
       "limit": 5
    }'
  )
)['results'] AS results;

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE CORTEX SEARCH SERVICE COSME_EC_HANDSON.ANALYTICS.REVIEW_SEARCH
    ON REVIEW_TEXT
    ATTRIBUTES REVIEW_ID, PRODUCT_ID, RATING, REVIEW_DATE, HELPFUL_COUNT
    WAREHOUSE = COSME_EC_WH
    TARGET_LAG = '1 hour'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT
        REVIEW_ID,
        PRODUCT_ID,
        RATING,
        REVIEW_DATE,
        HELPFUL_COUNT,
        REVIEW_TEXT
    FROM COSME_EC_HANDSON.RAW.REVIEWS
);

In [ ]:
%%sql -r review_search_result
-- REVIEW_SEARCH にクエリ（例：保湿に関するレビューを検索）
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'COSME_EC_HANDSON.ANALYTICS.REVIEW_SEARCH',
    '{
       "query": "保湿力が高い",
       "columns": ["REVIEW_ID", "PRODUCT_ID", "RATING", "REVIEW_TEXT"],
       "limit": 5
    }'
  )
)['results'] AS results;

In [ ]:
%%sql -r dataframe_15
-- Search Service の確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA COSME_EC_HANDSON.ANALYTICS;

## Section 3: Cortex Analyst — Semantic View

### セマンティックビューで定義すること

| 定義項目 | 何を定義する？ | 具体例 |
|---------|-------------|-------|
| テーブル | 分析対象のテーブルとその説明 | ORDERS, ORDER_ITEMS, PRODUCTS, CUSTOMERS |
| リレーションシップ | テーブル間の結合条件 | 注文ID = 明細の注文ID |
| ファクト | 集計対象の数値カラム | 売上金額、数量、単価、送料 |
| ディメンション | 分析の切り口となる属性 | 商品名、ブランド、カテゴリ、都道府県 |
| メトリクス | よく使う集計パターン | 総売上 = SUM(小計) |

### 4つのテーブルの関係

```
               CUSTOMER_ID                ORDER_ID               PRODUCT_ID
CUSTOMERS ◀──────────── ORDERS ◀──────────── ORDER_ITEMS ──────────▶ PRODUCTS
(顧客)                  (注文)                (注文明細)               (商品)
```


### カスタムインストラクション

セマンティックビューでは、テーブルやカラムの定義に加えて **カスタムインストラクション** を記述できます。これにより、SQL 生成の品質と一貫性が大きく向上します。

| インストラクション | 役割 |
|-----------------|------|
| `AI_SQL_GENERATION` | SQL 生成時のルールを指定する。数値フォーマット（小数点桁数・丸め）、デフォルトのソート順、フィルタ条件、NULL 処理など |
| `AI_QUESTION_CATEGORIZATION` | 対応可能な質問カテゴリと、対応 **できない** 質問を明示する。範囲外の質問には「対応できません」と返せるようになる |


### 検証済みクエリ（Verified Queries）

セマンティックビューには **検証済みクエリ** を登録できます。これは「この質問にはこの SQL を返す」という正解ペアで、Cortex Analyst の精度を確実に上げる仕組みです。

- Cortex Analyst の UI から **質問 → 生成された SQL を検証 → 保存** というフローで登録できます
- 登録した検証済みクエリは `VERIFIED_QUERIES` 句としてセマンティックビューに追加されます
- 似た質問が来たとき、検証済みクエリをベースに SQL を生成するため精度が向上します

> **ベストプラクティス**: まずセマンティックビューを作成し、Cortex Analyst で試しながら検証済みクエリを蓄積していくのが推奨ワークフローです。

### 3-1. レビュー分析セマンティックビュー（FACT_REVIEW_ANALYSIS）

レビューの星評価・観点別感情分析・顧客属性別の傾向を自然言語で分析可能にします。

対象テーブル: REVIEWS, REVIEW_DETAILS, PRODUCTS_WITH_CATEGORY, CUSTOMERS

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE SEMANTIC VIEW COSME_EC_HANDSON.ANALYTICS.FACT_REVIEW_ANALYSIS

    -- ========================================
    -- テーブル定義
    -- ========================================
    TABLES (
        REVIEWS AS COSME_EC_HANDSON.RAW.REVIEWS
            PRIMARY KEY (REVIEW_ID)
            COMMENT = '商品レビュー。星評価（1〜5）、レビュー本文、投稿日、「参考になった」数を含む。レビュー分析の起点テーブル。',

        REVIEW_DETAILS AS COSME_EC_HANDSON.ANALYTICS.REVIEW_DETAILS
            COMMENT = '観点別レビュー分析。各レビューを品質・効果、使用感・テクスチャー、価格・コスパ、肌トラブル・肌への影響、配送・梱包の5観点に分解し、観点ごとの感情ラベル・スコア・要約を格納。言及されていない観点は除外済み。',

        PRODUCTS AS COSME_EC_HANDSON.ANALYTICS.PRODUCTS_WITH_CATEGORY
            PRIMARY KEY (PRODUCT_ID)
            COMMENT = 'コスメ商品マスタ。商品名・ブランド・価格・説明文に加え、AIによる自動分類カテゴリ（スキンケア/メイク/ヘアケア/ボディケア/フレグランス）を含む。',

        CUSTOMERS AS COSME_EC_HANDSON.RAW.CUSTOMERS
            PRIMARY KEY (CUSTOMER_ID)
            COMMENT = '顧客マスタ。年代・肌質・都道府県・会員ランクなどの属性情報を管理。レビュー投稿者の属性分析に使用。'
    )

    -- ========================================
    -- リレーションシップ定義
    -- ========================================
    RELATIONSHIPS (
        DETAILS_REVIEWS   AS REVIEW_DETAILS(REVIEW_ID) REFERENCES REVIEWS(REVIEW_ID),
        REVIEWS_PRODUCTS  AS REVIEWS(PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID),
        REVIEWS_CUSTOMERS AS REVIEWS(CUSTOMER_ID) REFERENCES CUSTOMERS(CUSTOMER_ID)
    )

    -- ========================================
    -- ファクト定義（数値データ）
    -- ========================================
    FACTS (
        -- レビュー基本ファクト
        REVIEWS.rating_fact AS RATING
            COMMENT = '星評価（1〜5）。5が最高評価。',
        REVIEWS.helpful_count_fact AS HELPFUL_COUNT
            COMMENT = '「参考になった」の数。他の顧客がこのレビューを有用と評価した回数。',

        -- 観点別分析ファクト
        REVIEW_DETAILS.aspect_sentiment_score AS SENTIMENT_SCORE
            COMMENT = '観点ごとの感情スコア。positive=1, neutral=0, negative=-1。'
    )

    -- ========================================
    -- ディメンション定義（属性データ）
    -- ========================================
    DIMENSIONS (
        -- ========== REVIEWS ディメンション ==========
        REVIEWS.review_id AS REVIEW_ID
            COMMENT = 'レビューの一意識別子',
        REVIEWS.review_date AS REVIEW_DATE
            COMMENT = 'レビュー投稿日',
        REVIEWS.review_year AS YEAR(REVIEW_DATE)
            COMMENT = 'レビュー投稿年',
        REVIEWS.review_month AS MONTH(REVIEW_DATE)
            COMMENT = 'レビュー投稿月（1〜12）',
        REVIEWS.review_text AS REVIEW_TEXT
            COMMENT = 'レビュー本文',

        -- ========== REVIEW_DETAILS ディメンション ==========
        REVIEW_DETAILS.aspect_key AS ASPECT_KEY
            COMMENT = '分析観点の識別キー。quality/texture/price/skin_reaction/delivery の5種。',
        REVIEW_DETAILS.aspect_name AS ASPECT_NAME_JA
            COMMENT = '分析観点の日本語名。品質・効果/使用感・テクスチャー/価格・コスパ/肌トラブル・肌への影響/配送・梱包。',
        REVIEW_DETAILS.sentiment_label AS SENTIMENT_LABEL
            COMMENT = '観点ごとの感情ラベル。positive（肯定的）/negative（否定的）/neutral（中立的）。',
        REVIEW_DETAILS.aspect_summary AS ASPECT_SUMMARY
            COMMENT = '観点ごとの1文要約（日本語）',

        -- ========== PRODUCTS ディメンション ==========
        PRODUCTS.product_name AS PRODUCT_NAME
            COMMENT = '商品名',
        PRODUCTS.brand AS BRAND
            COMMENT = 'ブランド名',
        PRODUCTS.product_price AS PRICE
            COMMENT = '商品の税込定価（円）',
        PRODUCTS.category AS CATEGORY
            COMMENT = '商品カテゴリ。スキンケア/メイク/ヘアケア/ボディケア/フレグランス の5種。',
        PRODUCTS.is_organic AS IS_ORGANIC
            COMMENT = 'オーガニック認証の有無（TRUE/FALSE）',

        -- ========== CUSTOMERS ディメンション ==========
        CUSTOMERS.customer_name AS CUSTOMER_NAME
            COMMENT = '顧客名',
        CUSTOMERS.age_group AS AGE_GROUP
            COMMENT = '顧客の年代。10代/20代/30代/40代/50代以上 の5区分。',
        CUSTOMERS.prefecture AS PREFECTURE
            COMMENT = '顧客の居住都道府県',
        CUSTOMERS.membership_rank AS MEMBERSHIP_RANK
            COMMENT = '会員ランク。レギュラー/シルバー/ゴールド/プラチナ の4段階。'
    )

    -- ========================================
    -- メトリクス定義（集計関数）
    -- ========================================
    METRICS (
        -- レビュー基本メトリクス
        REVIEWS.total_reviews AS COUNT(REVIEWS.review_id)
            COMMENT = 'レビューの総件数',
        REVIEWS.avg_rating AS AVG(REVIEWS.rating_fact)
            COMMENT = '星評価の平均（1〜5）',
        REVIEWS.total_helpful AS SUM(REVIEWS.helpful_count_fact)
            COMMENT = '「参考になった」の合計数',
        REVIEWS.avg_helpful AS AVG(REVIEWS.helpful_count_fact)
            COMMENT = '「参考になった」の平均数',

        -- 観点別メトリクス
        REVIEW_DETAILS.avg_aspect_sentiment AS AVG(REVIEW_DETAILS.aspect_sentiment_score)
            COMMENT = '観点ごとの感情スコアの平均',
        REVIEW_DETAILS.total_aspect_mentions AS COUNT(REVIEW_DETAILS.aspect_key)
            COMMENT = '各観点が言及された回数',

        -- ユニーク数メトリクス
        REVIEWS.unique_reviewers AS COUNT(DISTINCT REVIEWS.customer_id)
            COMMENT = 'レビューを投稿したユニーク顧客数',
        REVIEWS.unique_reviewed_products AS COUNT(DISTINCT REVIEWS.product_id)
            COMMENT = 'レビューが投稿されたユニーク商品数'
    )

    -- ========================================
    -- コメント・カスタムインストラクション
    -- ========================================
    COMMENT = 'コスメECサイトのレビュー分析セマンティックビュー。星評価と感情スコアの相関分析、商品別・ブランド別・カテゴリ別のレビュー傾向分析、年代別・肌質別の顧客レビュー傾向分析、観点別（品質・使用感・価格・肌トラブル・配送）のポジネガ分析が可能。'

    AI_SQL_GENERATION $$
このセマンティックビューはコスメECサイトのレビュー分析用です。以下のルールに従ってSQLを生成してください。

数値フォーマット:
- 平均評価や感情スコアは小数点第2位まで丸めること（ROUND(..., 2)）。
- 件数や合計は整数で表示すること。

デフォルトフィルタ:
- 特に日付の指定がない場合は全期間のデータを対象にすること。

観点別分析のルール:
- REVIEW_DETAILSテーブルは1レビューに対して複数行（最大5観点）が存在する。
- 観点別に集計する場合は必ずASPECT_KEYまたはASPECT_NAME_JAでグループ化すること。
- 観点の種類はquality（品質・効果）、texture（使用感・テクスチャー）、price（価格・コスパ）、skin_reaction（肌トラブル・肌への影響）、delivery（配送・梱包）の5種。

感情分析のルール:
- SENTIMENT_LABELはpositive/negative/neutralの3値。
- SENTIMENT_SCOREはpositive=1, neutral=0, negative=-1の数値。
- ポジティブ率を求める場合はpositive件数/全件数*100で計算すること。

ソート:
- ランキング系の質問では件数や平均値の降順でソートすること。
- 日付に関する質問では時系列順（昇順）でソートすること。
$$

    AI_QUESTION_CATEGORIZATION $$
対応可能な質問カテゴリ:
- レビューの星評価に関する分析（平均評価、評価分布、高評価・低評価の傾向）
- 観点別の感情分析（品質・使用感・価格・肌トラブル・配送に関するポジネガ傾向）
- 商品別・ブランド別・カテゴリ別のレビュー分析
- 顧客属性別（年代・会員ランク・都道府県）のレビュー傾向
- 参考になった数に基づくレビュー品質分析
- 時系列でのレビュー傾向分析

対応できない質問:
- 注文金額・売上・在庫に関する質問は注文分析のセマンティックビュー（FACT_ORDER_ANALYSIS）をご利用ください、と案内すること。
- 個人情報の詳細（メールアドレス等）に関する質問は個人情報に関する質問にはお答えできません、と回答すること。

曖昧な質問への対応:
- レビュー分析してのような曖昧な質問には商品別の平均評価と観点別の感情スコアを返すこと。
- どの商品が人気のような質問にはレビュー件数と平均評価の両方を返すこと。
$$;

In [ ]:
%%sql -r sv_review_result
-- Semantic View (REVIEW) を SQL で直接クエリ
-- 観点別の平均感情スコア（AGG関数使用）
SELECT
  ASPECT_NAME,
  AGG(AVG_ASPECT_SENTIMENT) AS AVG_SENTIMENT
FROM COSME_EC_HANDSON.ANALYTICS.FACT_REVIEW_ANALYSIS
GROUP BY ASPECT_NAME
ORDER BY AVG_SENTIMENT DESC;

### 3-2. 注文分析セマンティックビュー（FACT_ORDER_ANALYSIS）

注文・売上・在庫データを自然言語で分析可能にします。INVENTORYテーブルも含めて在庫アラート分析にも対応します。

対象テーブル: ORDERS, ORDER_ITEMS, PRODUCTS, CUSTOMERS, INVENTORY

In [ ]:
%%sql -r dataframe_17
CREATE OR REPLACE SEMANTIC VIEW COSME_EC_HANDSON.ANALYTICS.FACT_ORDER_ANALYSIS

    -- ========================================
    -- テーブル定義
    -- ========================================
    TABLES (
        ORDERS AS COSME_EC_HANDSON.RAW.ORDERS
            PRIMARY KEY (ORDER_ID)
            COMMENT = '注文ヘッダ。注文ステータス（注文確定/決済完了/出荷準備中/配送中/配送済み/キャンセル/返品）・合計金額・送料・支払方法・配送先都道府県を管理。',

        ORDER_ITEMS AS COSME_EC_HANDSON.RAW.ORDER_ITEMS
            PRIMARY KEY (ORDER_ITEM_ID)
            COMMENT = '注文明細。注文ごとの商品・数量・購入時単価・小計（数量×単価）を管理。商品レベルの売上分析にはこのテーブルを使用する。',

        PRODUCTS AS COSME_EC_HANDSON.RAW.PRODUCTS
            PRIMARY KEY (PRODUCT_ID)
            COMMENT = 'コスメ商品マスタ。商品名・ブランド・税込定価・商品説明・発売日・オーガニック認証有無を含む。',

        CUSTOMERS AS COSME_EC_HANDSON.RAW.CUSTOMERS
            PRIMARY KEY (CUSTOMER_ID)
            COMMENT = '顧客マスタ。年代・肌質・都道府県・会員ランクなどの属性情報を管理。顧客セグメント別の購買分析に使用。',

        INVENTORY AS COSME_EC_HANDSON.RAW.INVENTORY
            PRIMARY KEY (PRODUCT_ID)
            COMMENT = '在庫テーブル。商品ごとの現在在庫数・発注点（この数量を下回ったら補充が必要）・保管倉庫・最終入荷日を管理。'
    )

    -- ========================================
    -- リレーションシップ定義
    -- ========================================
    RELATIONSHIPS (
        ORDERS_CUSTOMERS AS ORDERS(CUSTOMER_ID) REFERENCES CUSTOMERS(CUSTOMER_ID),
        ITEMS_ORDERS     AS ORDER_ITEMS(ORDER_ID) REFERENCES ORDERS(ORDER_ID),
        ITEMS_PRODUCTS   AS ORDER_ITEMS(PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID),
        INVENTORY_PRODUCTS AS INVENTORY(PRODUCT_ID) REFERENCES PRODUCTS(PRODUCT_ID)
    )

    -- ========================================
    -- ファクト定義（数値データ）
    -- ========================================
    FACTS (
        -- 注文ファクト
        ORDERS.total_amount_fact AS TOTAL_AMOUNT
            COMMENT = '注文の合計金額（円、税込）。送料を含む場合がある。',
        ORDERS.shipping_fee_fact AS SHIPPING_FEE
            COMMENT = '送料（円）。5,500円以上の注文は送料無料（0円）。',

        -- 注文明細ファクト
        ORDER_ITEMS.quantity_fact AS QUANTITY
            COMMENT = '購入数量',
        ORDER_ITEMS.unit_price_fact AS UNIT_PRICE
            COMMENT = '購入時の商品単価（円）',
        ORDER_ITEMS.subtotal_fact AS SUBTOTAL
            COMMENT = '明細小計（数量×単価）。商品レベルの売上分析にはこのカラムを使用する。',

        -- 商品ファクト
        PRODUCTS.price_fact AS PRICE
            COMMENT = '商品の税込定価（円）',

        -- 在庫ファクト
        INVENTORY.stock_quantity_fact AS STOCK_QUANTITY
            COMMENT = '現在の在庫数',
        INVENTORY.reorder_point_fact AS REORDER_POINT
            COMMENT = '発注点。在庫数がこの値を下回ったら補充が必要。'
    )

    -- ========================================
    -- ディメンション定義（属性データ）
    -- ========================================
    DIMENSIONS (
        -- ========== ORDERS ディメンション ==========
        ORDERS.order_id_dim AS ORDER_ID
            COMMENT = '注文の一意識別子',
        ORDERS.customer_id_dim AS CUSTOMER_ID
            COMMENT = '顧客ID',
        ORDERS.order_date_dim AS ORDER_DATE
            COMMENT = '注文日',
        ORDERS.order_year AS YEAR(ORDER_DATE)
            COMMENT = '注文年',
        ORDERS.order_month AS MONTH(ORDER_DATE)
            COMMENT = '注文月（1〜12）',
        ORDERS.order_status_dim AS ORDER_STATUS
            COMMENT = '注文ステータス。注文確定/決済完了/出荷準備中/配送中/配送済み/キャンセル/返品 の7種。',
        ORDERS.payment_method_dim AS PAYMENT_METHOD
            COMMENT = '支払方法。クレジットカード/電子マネー/コンビニ払い/代引き の4種。',
        ORDERS.shipping_prefecture_dim AS SHIPPING_PREFECTURE
            COMMENT = '配送先の都道府県',

        -- ========== ORDER_ITEMS ディメンション ==========
        ORDER_ITEMS.order_item_id AS ORDER_ITEM_ID
            COMMENT = '注文明細の一意識別子',

        -- ========== PRODUCTS ディメンション ==========
        PRODUCTS.product_name_dim AS PRODUCT_NAME
            COMMENT = '商品名',
        PRODUCTS.brand_dim AS BRAND
            COMMENT = 'ブランド名',
        PRODUCTS.is_organic_dim AS IS_ORGANIC
            COMMENT = 'オーガニック認証の有無（TRUE/FALSE）',
        PRODUCTS.launch_date_dim AS LAUNCH_DATE
            COMMENT = '商品の発売日',

        -- ========== CUSTOMERS ディメンション ==========
        CUSTOMERS.customer_name_dim AS CUSTOMER_NAME
            COMMENT = '顧客名',
        CUSTOMERS.age_group_dim AS AGE_GROUP
            COMMENT = '顧客の年代。10代/20代/30代/40代/50代以上 の5区分。',
        CUSTOMERS.prefecture_dim AS PREFECTURE
            COMMENT = '顧客の居住都道府県',
        CUSTOMERS.membership_rank_dim AS MEMBERSHIP_RANK
            COMMENT = '会員ランク。レギュラー/シルバー/ゴールド/プラチナ の4段階。',

        -- ========== INVENTORY ディメンション ==========
        INVENTORY.warehouse_location_dim AS WAREHOUSE_LOCATION
            COMMENT = '保管倉庫。東京倉庫A棟/大阪倉庫B棟/福岡倉庫C棟。',
        INVENTORY.last_restocked_at_dim AS LAST_RESTOCKED_AT
            COMMENT = '最終入荷日'
    )

    -- ========================================
    -- メトリクス定義（集計関数）
    -- ========================================
    METRICS (
        -- 売上メトリクス
        total_revenue AS SUM(ORDER_ITEMS.subtotal_fact)
            COMMENT = '商品売上の合計（円）。注文明細の小計を合算。',
        avg_order_value AS AVG(ORDERS.total_amount_fact)
            COMMENT = '注文1件あたりの平均金額（円）。客単価。',
        total_orders AS COUNT(DISTINCT ORDERS.order_id_dim)
            COMMENT = '注文の総件数',
        total_items_sold AS SUM(ORDER_ITEMS.quantity_fact)
            COMMENT = '販売した商品の総数量',
        avg_unit_price AS AVG(ORDER_ITEMS.unit_price_fact)
            COMMENT = '購入時単価の平均（円）',
        total_shipping_fees AS SUM(ORDERS.shipping_fee_fact)
            COMMENT = '送料の合計（円）',

        -- 顧客メトリクス
        unique_customers AS COUNT(DISTINCT ORDERS.customer_id_dim)
            COMMENT = '購入した顧客のユニーク数',

        -- 在庫メトリクス
        total_stock AS SUM(INVENTORY.stock_quantity_fact)
            COMMENT = '在庫数の合計',
        avg_stock AS AVG(INVENTORY.stock_quantity_fact)
            COMMENT = '商品あたりの平均在庫数'
    )

    -- ========================================
    -- コメント・カスタムインストラクション
    -- ========================================
    COMMENT = 'コスメECサイトの注文・売上・在庫分析セマンティックビュー。売上トレンド分析、商品別・ブランド別の売上ランキング、顧客セグメント別（年代・会員ランク）の購買傾向分析、注文ステータス別の分析、在庫状況の確認と発注点アラートが可能。'

    AI_SQL_GENERATION $$
このセマンティックビューはコスメECサイトの注文・売上・在庫分析用です。以下のルールに従ってSQLを生成してください。

数値フォーマット:
- 金額は整数で表示すること（円単位、小数点なし）。
- 平均金額は小数点第0位まで丸めること（ROUND(..., 0)）。
- 割合やパーセンテージは小数点第1位まで丸めること（ROUND(..., 1)）。

デフォルトフィルタ:
- 特に日付の指定がない場合は全期間のデータを対象にすること。
- キャンセル・返品を除外する分析の場合はORDER_STATUSでフィルタすること。

売上分析のルール:
- 商品レベルの売上分析にはORDER_ITEMSテーブルのSUBTOTAL（明細小計）を使用すること。
- 注文レベルの売上分析にはORDERSテーブルのTOTAL_AMOUNT（注文合計）を使用すること。
- 送料はSHIPPING_FEEで別管理。

在庫分析のルール:
- 在庫切れはSTOCK_QUANTITY=0で判定すること。
- 補充が必要な商品はSTOCK_QUANTITY<REORDER_POINTで判定すること。
- 在庫分析は商品単位で行うこと（PRODUCT_NAMEでグループ化）。

ソート:
- 売上ランキングでは金額の降順でソートすること。
- 日付に関する質問では時系列順（昇順）でソートすること。
- 在庫アラートでは在庫数の昇順でソートすること。
$$

    AI_QUESTION_CATEGORIZATION $$
対応可能な質問カテゴリ:
- 売上分析（総売上、平均注文額、商品別・ブランド別売上、時系列トレンド）
- 注文分析（注文件数、注文ステータス別集計、支払方法別分析）
- 顧客分析（年代別・会員ランク別・都道府県別の購買傾向、客単価）
- 商品分析（商品別販売数量、ブランド別売上ランキング）
- 在庫分析（在庫状況、在庫切れ商品、補充が必要な商品、倉庫別在庫）
- 配送分析（配送先都道府県別の注文傾向、送料分析）

対応できない質問:
- レビュー内容・星評価・感情分析に関する質問はレビュー分析のセマンティックビュー（FACT_REVIEW_ANALYSIS）をご利用ください、と案内すること。
- 個人情報の詳細（メールアドレス等）に関する質問は個人情報に関する質問にはお答えできません、と回答すること。

曖昧な質問への対応:
- 売上を教えてのような曖昧な質問には月別の売上推移と商品別売上トップ10を返すこと。
- 在庫は大丈夫のような質問には在庫切れ商品と補充が必要な商品のリストを返すこと。
$$;

In [ ]:
%%sql -r sv_order_result
-- Semantic View (ORDER) を SQL で直接クエリ
-- ブランド別売上ランキング（AGG関数使用）
SELECT
  BRAND_DIM AS BRAND,
  AGG(TOTAL_REVENUE) AS TOTAL_SALES
FROM COSME_EC_HANDSON.ANALYTICS.FACT_ORDER_ANALYSIS
GROUP BY BRAND_DIM
ORDER BY TOTAL_SALES DESC
LIMIT 10;

## Section 4: Cortex Agent（AI エージェント）

いよいよエージェントを組み立てます。

Section 2 で作った **3 つの Cortex Search** と Section 3 で作った **2 つの Cortex Analyst** を組み合わせて、5 つのツールを持つ AI エージェントを構築します。

### エージェントの仕組み

```
ユーザーの質問
     │
     ▼
┌─────────────────────────────────────────────┐
│  ① 質問の意図を理解                           │
│     LLM が質問を解釈し、どのツールが最適かを判断  │
├─────────────────────────────────────────────┤
│  ② ツール選択・実行                           │
│     orchestration の指示に従い 1 つ以上の       │
│     ツールを選択して実行                        │
├─────────────────────────────────────────────┤
│  ③ 結果の統合・回答生成                        │
│     ツールの実行結果を統合し、response の指示に   │
│     従って自然言語で回答を生成                   │
└─────────────────────────────────────────────┘
```

### ツールの使い分け

| 質問の種類 | 使われるツール | 例 |
|-----------|-------------|---|
| FAQ・返品ポリシー・配送 | FAQ_SEARCH | 「返品ポリシーを教えて」 |
| 商品検索・おすすめ | PRODUCT_SEARCH | 「敏感肌向けスキンケア商品は？」 |
| レビュー内容の確認 | REVIEW_SEARCH | 「保湿力が高い口コミがある商品は？」 |
| 売上・注文・在庫の数値分析 | ORDER_ANALYST | 「先月の売上トップ5は？」 |
| 星評価・感情分析の集計 | REVIEW_ANALYST | 「品質の観点でポジティブ率が高いブランドは？」 |
| 複合的な質問 | Search → Analyst | 「保湿系商品の売上推移は？」 |

### Cortex Agent の作成

3 つの Search + 2 つの Analyst を束ねる Agent を作成します。

In [ ]:
%%sql -r dataframe_20
CREATE OR REPLACE AGENT COSME_EC_HANDSON.ANALYTICS.COSME_ANALYST_AGENT
    COMMENT = 'コスメECサイトの商品・注文・レビューを横断的に分析するAIエージェント'
FROM SPECIFICATION $$
{
  "instructions": {
    "response": "あなたはコスメECサイト「Cosme EC」の高度な分析アシスタントです。5つの専門ツールを使い分けて、ユーザーの質問に日本語で丁寧に回答してください。数値データは適切にフォーマットしてください（金額は円表記でカンマ区切り、比率は%表記、平均値は小数点第2位まで）。分析結果には根拠となるデータを含めてください。複数のツールの結果を組み合わせて、包括的な回答を提供してください。",
    "orchestration": "ツールの使い分けルール:\n\n(1) FAQ・返品ポリシー・配送・会員制度・スキンケア相談\n    → FAQ_SEARCH を使用\n\n(2) 商品の検索・おすすめ・商品情報の確認\n    → PRODUCT_SEARCH を使用\n\n(3) レビュー内容の検索・特定の感想や口コミの確認\n    → REVIEW_SEARCH を使用\n\n(4) 売上・注文件数・客単価・在庫などの数値分析\n    → ORDER_ANALYST を使用\n\n(5) 星評価の集計・観点別の感情分析・レビュー傾向の数値分析\n    → REVIEW_ANALYST を使用\n\n複合的な質問の場合:\nまず Search で情報を特定し、次に Analyst で数値分析してください。\n\n例1: 「保湿系商品の売上推移」\n  → PRODUCT_SEARCH で保湿系商品を特定\n  → ORDER_ANALYST で売上分析\n\n例2: 「肌荒れに関するレビューが多い商品」\n  → REVIEW_SEARCH で肌荒れレビューを検索\n  → REVIEW_ANALYST で集計分析",
    "sample_questions": [
      {
        "question": "返品・交換のポリシーについて教えてください。"
      },
      {
        "question": "敏感肌向けのスキンケア商品を探しています。"
      },
      {
        "question": "保湿クリームのレビューで評判の良いものを教えて。"
      },
      {
        "question": "先月の売上トップ5の商品を教えてください。"
      },
      {
        "question": "品質・効果の観点でポジティブ評価が多いブランドはどこですか？"
      },
      {
        "question": "オーガニック商品の売上は全体の何パーセントですか？"
      }
    ]
  },
  "tools": [
    {
      "tool_spec": {
        "type": "cortex_analyst_text_to_sql",
        "name": "ORDER_ANALYST",
        "description": "注文・売上・在庫データの数値分析ツール。売上金額の集計、注文件数、客単価、商品別・ブランド別の売上ランキング、月別トレンド、在庫状況の確認、顧客セグメント別（年代・会員ランク・都道府県）の購買傾向分析に使用します。金額や数量の集計が必要な質問にはこのツールを使ってください。"
      }
    },
    {
      "tool_spec": {
        "type": "cortex_analyst_text_to_sql",
        "name": "REVIEW_ANALYST",
        "description": "レビュー・評価データの数値分析ツール。星評価の平均・分布、観点別（品質・使用感・価格・肌トラブル・配送）の感情スコア集計、商品別・ブランド別・カテゴリ別のレビュー傾向分析、ポジティブ率・ネガティブ率の計算に使用します。レビューの数値的な集計や統計が必要な質問にはこのツールを使ってください。"
      }
    },
    {
      "tool_spec": {
        "type": "cortex_search",
        "name": "FAQ_SEARCH",
        "description": "FAQナレッジベースの検索ツール。返品・交換ポリシー、配送情報、成分・アレルギー情報、スキンケア相談、ポイント・会員制度に関する質問にはこのツールを使ってください。"
      }
    },
    {
      "tool_spec": {
        "type": "cortex_search",
        "name": "PRODUCT_SEARCH",
        "description": "商品情報の検索ツール。商品名・ブランド・カテゴリ・商品説明文で商品を意味検索します。特定の商品を探したい、おすすめ商品を知りたい、商品の詳細を確認したい場合にこのツールを使ってください。"
      }
    },
    {
      "tool_spec": {
        "type": "cortex_search",
        "name": "REVIEW_SEARCH",
        "description": "レビュー本文の検索ツール。レビューの内容を意味検索します。特定の商品や成分に対する口コミ・感想を探したい場合にこのツールを使ってください。数値的な集計ではなく、具体的なレビュー内容を確認したい場合に適しています。"
      }
    }
  ],
  "tool_resources": {
    "ORDER_ANALYST": {
      "semantic_view": "COSME_EC_HANDSON.ANALYTICS.FACT_ORDER_ANALYSIS",
      "execution_environment": {"type": "warehouse", "warehouse": "COSME_EC_WH"}
    },
    "REVIEW_ANALYST": {
      "semantic_view": "COSME_EC_HANDSON.ANALYTICS.FACT_REVIEW_ANALYSIS",
      "execution_environment": {"type": "warehouse", "warehouse": "COSME_EC_WH"}
    },
    "FAQ_SEARCH": {
      "name": "COSME_EC_HANDSON.ANALYTICS.FAQ_SEARCH",
      "max_results": 5
    },
    "PRODUCT_SEARCH": {
      "name": "COSME_EC_HANDSON.ANALYTICS.PRODUCT_SEARCH",
      "max_results": 10
    },
    "REVIEW_SEARCH": {
      "name": "COSME_EC_HANDSON.ANALYTICS.REVIEW_SEARCH",
      "max_results": 10
    }
  }
}
$$;

In [ ]:
%%sql -r dataframe_22
SELECT
  TRY_PARSE_JSON(
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
      'COSME_EC_HANDSON.ANALYTICS.COSME_ANALYST_AGENT',
      $${
        "messages": [
          {
            "role": "user",
            "content": [
              {
                "type": "text",
                "text": "先月の売上トップ5の商品を教えてください。"
              }
            ]
          }
        ]
      }$$
    )
  ) AS RESP;

## Section 5:　Snowflake Intelligence

エージェントが完成しました！  
Snowflake Intelligence から動かしてみよう！

### 試してみたい質問例

**売上分析（ORDER_ANALYST）**
- 「先月の売上トップ5の商品を教えてください。」
- 「月別の売上推移を教えてください。」
- 「ブランド別の売上ランキングを教えてください。」
- 「オーガニック商品の売上は全体の何パーセントですか？」

**レビュー分析（REVIEW_ANALYST）**
- 「品質・効果の観点でポジティブ評価が多いブランドはどこですか？」
- 「星1のレビューが多い商品はどれ？」
- 「観点別の感情スコアを比較してください。」

**FAQ検索（FAQ_SEARCH）**
- 「返品・交換のポリシーについて教えてください。」
- 「ポイント制度について教えてください。」

**商品検索（PRODUCT_SEARCH）**
- 「敏感肌向けのスキンケア商品を探しています。」
- 「オーガニック認証の商品にはどんなものがある？」

**レビュー検索（REVIEW_SEARCH）**
- 「保湿クリームのレビューで評判の良いものを教えて。」
- 「肌荒れしたというレビューがある商品を教えて。」

**複合的な質問（Search + Analyst）**
- 「敏感肌向け商品の売上推移を月別で見せてください。」
- 「スキンケアカテゴリの売上トップ3と、それぞれのレビュー傾向を教えて。」


## Section 6: Cortex Agent Evaluations（エージェント評価）

エージェントの品質を定量的に測定するためのフレームワークです。

### 評価の流れ

```
1. 評価データセット作成    →  Ground Truth（正解データ）を含む質問セット
2. YAML 設定ファイル確認   →  メトリクス定義（ビルトイン + カスタム LLM Judge）
3. 評価実行              →  EXECUTE_AI_EVALUATION でエージェントを評価
4. 結果確認              →  GET_AI_EVALUATION_DATA または Snowsight UI で確認
```

### 評価メトリクス

| メトリクス | 種別 | 説明 |
|-----------|------|------|
| Answer Correctness | ビルトイン | 回答の正確性 |
| Logical Consistency | ビルトイン | 回答の論理的一貫性 |
| Groundedness | カスタム | 回答がツール出力のエビデンスに基づいているか |
| Execution Efficiency | カスタム | ツール呼び出しの効率性（冗長な呼び出しがないか） |
| Tool Selection | カスタム | 適切なツールが選択されているか |

参考: [Getting Started with Cortex Agent Evaluations](https://www.snowflake.com/en/developers/guides/getting-started-with-cortex-agent-evaluations/)

### 6-1. 評価データセットの作成

`setup.sql` の STEP 19 で作成した `EVALS_TABLE`（21件の Ground Truth データ）から、Cortex Agent Evaluations 用のデータセットを作成します。

In [ ]:
%%sql -r eval_create_dataset
USE DATABASE COSME_EC_HANDSON;
USE SCHEMA ANALYTICS;
CALL SYSTEM$CREATE_EVALUATION_DATASET(
    'Cortex Agent',
    'COSME_EC_HANDSON.ANALYTICS.EVALS_TABLE',
    'COSME_EC_HANDSON.ANALYTICS.COSME_AGENT_EVALSET',
    OBJECT_CONSTRUCT('query_text', 'INPUT_QUERY', 'expected_tools', 'GROUND_TRUTH_DATA')
);

In [ ]:
%%sql -r eval_show_datasets
SHOW DATASETS;

### 6-2. YAML 評価設定ファイルの確認

`setup.sql` の STEP 20 でアップロード済みの YAML 設定ファイルを確認します。
この YAML にはビルトインメトリクス（answer_correctness, logical_consistency）とカスタムメトリクス（groundedness, execution_efficiency, tool_selection）が定義されています。

In [ ]:
%%sql -r eval_ls_config
LS @COSME_EC_HANDSON.ANALYTICS.EVAL_CONFIG_STAGE;

### 6-3. 評価の実行

評価を開始します。通常 3〜5 分かかります。

In [ ]:
%%sql -r eval_start_baseline
USE DATABASE COSME_EC_HANDSON;
USE SCHEMA ANALYTICS;
CALL EXECUTE_AI_EVALUATION(
    'START',
    OBJECT_CONSTRUCT('run_name', 'COSME_AGENT_BASELINE_RUN'),
    '@COSME_EC_HANDSON.ANALYTICS.EVAL_CONFIG_STAGE/cosme_agent_eval_config.yaml'
);

### 6-4. 評価結果の確認

In [ ]:
%%sql -r eval_baseline_results
SELECT * FROM TABLE(SNOWFLAKE.LOCAL.GET_AI_EVALUATION_DATA(
    'COSME_EC_HANDSON',
    'ANALYTICS',
    'COSME_ANALYST_AGENT',
    'cortex agent',
    'COSME_AGENT_BASELINE_RUN'
));

### 評価結果の確認方法

Snowsight UI でも評価結果を確認できます：

**AI & ML > Agents > COSME_ANALYST_AGENT > Evaluations タブ**

各レコードをクリックすると、エージェントトレース（ツール呼び出しの詳細）と各メトリクスのスコアを確認できます。

> 評価結果を基にエージェントを改善し再評価する方法については `eval/README.md` を参照してください。